---
title: SymPy
subtitle: Obliczenia symboliczne z pakietem SymPy
author:
  - name: Krzysztof Molenda
    affiliations: URK/WIPiE/KIPLiIS
    email: krzysztof.molenda@urk.edu.pl
date: 2026/03/02
---

---


<img src="https://docs.sympy.org/latest/_static/sympylogo.png"
     alt="SymPy"
     style="float:right; margin:0 1em 1em 0; width:150px;">

[Sympy](http://www.sympy.org/en/index.html) jest opisana jako:

> "... Python library for symbolic mathematics."

Oznacza to, że może być wykorzystana, między innymi do:

- tworzenia i modyfikowania symbolicznych wyrażeń matematycznych;
- symbolicznego rozwiązywania równań (i układów równań);
- wykonywania operacji rachunku różniczkowego (granice, pochodne, całki, równania różniczkowe), stosując symboliczne przekształcenia;
- rysowania funkcji opisanych symbolicznie.

Ma jeszcze wiele więcej możliwości, z którymi można zapoznać się na stronie: <http://www.sympy.org/en/index.html>

<div style="clear:both;"></div>

## Instalacja biblioteki SymPy

Aby móc wykonywać obliczenia symboliczne, należy w swoim środowisku Python + Jupyter Notebook zainstalować bibliotekę `sympy`, o ile nie jest ona zainstalowana domyślnie.

In [ ]:
# %mamba install sympy
# %pip install sympy

W kolejnym kroku importujemy bibliotekę do notatnika, nadając jej alias `sym`. 

In [ ]:
import sympy as sym # alias sym
print(sym.__version__)

Od tego momentu elementy z biblioteki (stałe, funkcje, metody, ...) poprzedzać będziemy zapisem `sym.`, na przykład `sym.sqrt(2)` aby wprowadzić $\sqrt 2$ lub `sym.Rational(1, 2)` aby zaprezentować ułamek $\frac{1}{2}$.

Importujemy dodatkowe biblioteki potencjalnie wykorzystywane w notatniku, wprowadzamy podstawową konfigurację biblioteki SymPy obowiązującą dla całego notatnika.

In [ ]:
import matplotlib
import numpy as np

sym.init_printing() # włącza ładniejsze renderowanie wzorów

:::{tip} Porada
SymPy pozwala generować kod zapisu $\LaTeX$ poleceniem

```python
sympy.latex(expr)
```

gdzie `expr` jest wyrażeniem symbolicznym.

:::

In [ ]:
x = sym.symbols('x')
expr = x**3 - 3 * x**2 + sym.sqrt(2)/(x-1)
display(expr)          # wyświetla wzór w notatniku
print(expr)            # wypisuje wzór w formacie liniowym dla Pythona
print(sym.latex(expr)) # wypisuje kod wzoru w LaTeX

## Symbole, wyrażenia, ewaluacja

SymPy działa jak **system algebry symbolicznej** (CAS, _Computer algebra system_) zapisany w Pythonie - zamiast od razu liczyć przybliżone liczby, buduje dokładne obiekty matematyczne i wykonuje na nich przekształcenia algebraiczne, różniczkowanie, całkowanie, rozwiązywanie równań czy operacje na macierzach. Najważniejsze jest to, że w SymPy wyrażenie jest reprezentacją matematyczną, a nie tylko chwilowym wynikiem obliczenia numerycznego.

W SymPy zmienne matematyczne nie są liczbami, lecz symbolam. Jeśli będziemy chcieli wprowadzić wyrażenie symboliczne, **musimy** zadeklarować wykorzystane w nim symbole (zmienne):

In [ ]:
x = sym.symbols('x') # deklaracja symbolu x jako zmiennej x

Teraz możemy wprowadzić wyrażenie $x^2 - x$ w formacie liniowym Pythona:

In [ ]:
expr = x**2 - x
display(expr) # wyświetlanie wyrażenia w LaTeX
print(expr)   # wypisywanie wyrażenia - format liniowy

Symbol `x` nie ma wartości - jest abstrakcyjnym obiektem algebraicznym.

Dzięki temu:

- `x + x → 2*x`
- `x * x → x**2`
- `sin(x + y)` pozostaje funkcją symboliczną

SymPy traktuje symbole jak elementy drzewa wyrażeń.

Wyrażenie w SymPy to drzewo operacji (_expression tree_).

**Przykład:**

```python
expr = (x + 1)**2
```

`expr` to nie jest liczba - to struktura:

```text
       **
     /    \
    +      2
  /   \
 x     1
```

Dlatego SymPy może:
- upraszczać (`simplify(expr)`)
- rozwijać (`expand(expr)`)
- różniczkować (`diff(expr, x)`)
- całkować (`integrate(expr, x)`)
- podstawiać (`expr.subs(x, 3)`)

**Eksperyment:**

Sprawdźmy, czy poniższe wyrażenie jest prawdziwe:

$$(a + b) ^ 2 = a ^ 2 + 2ab + b ^2$$

Najpierw, zadeklarować musimy zmienne symboliczne $a, b$:

In [ ]:
a, b = sym.symbols('a, b')

Teraz zapiszemy przy ich pomocy wyrażenie:

In [ ]:
expr1 = (a + b)**2
display(expr1)

Wyrażenie to możemy rozwinąć (`expand`):

In [ ]:
expr2 = expr1.expand()
display(expr2)

Zatem pokazaliśmy, że $(a+b)^2$ po rozwinięciu równe jest $a^{2} + 2 a b + b^{2}$.

Sprawdźmy to dosłownie:

In [ ]:
# Czy expr1 == expr2 ?
expr1 == expr2

Zaskoczenie, ponieważ spodziewaliśmy się `True`.

:::{important} Wyjaśnienie

Mechanizm SymPy można rozumieć jako budowanie drzewa wyrażeń, gdzie każdy wzór składa się z mniejszych obiektów, takich jak suma, iloczyn, potęga symbol liczby lub zmiennej. Na przykład wyrażenie $(x+1)^2$ ma inną strukturę niż $x^2+2x+1$, nawet jeśli matematycznie są równe, dlatego SymPy domyślnie nie traktuje ich jako identycznych zapisów.
:::

Jedna z najczęstszych pułapek polega na tym, że `==` w SymPy sprawdza dokładną równość strukturalną, a nie symboliczną równość matematyczną. Dlatego zapis `(x + 1)**2 == x**2 + 2*x + 1` daje `False`, ponieważ te dwa wyrażenia mają inną budowę wewnętrzną.

Zatem, jeśli chcemy porównać dwa wyrażenia (równość matematyczna), stosujemy `simplify(expr1 - expr2) == 0`.

In [ ]:
sym.simplify(expr1 - expr2) == 0

:::{admonition} Ćwiczenie 1

Zweryfikuj prawdziwość poniższych równań:

- $(a - b) ^ 2 = a ^ 2 - 2 a b + b^2$
- $a ^ 2 - b ^ 2 = (a - b) (a + b)$ (Podpowiedź: zamiast `expand`, użyj `factor` - rozkład na iloczyn)
:::

In [ ]:
# wpisz kod

### Ewaluacja

Jeżeli w Pythonie napiszesz:

```python
x = 2
x + 1 # daje 3
```

W SymPy:

```python
x = sym.symbols('x')
x + 1 # daje x + 1
```

SymPy **nie oblicza**, dopóki nie każesz mu obliczyć. Ewaluacja w SymPy ma dwa tryby:

- symboliczna -- manipulacja strukturą wyrażenia (np. upraszczanie, różniczkowanie).
- numeryczna -- `subs` - podstawienie, `evalf` - przeliczenie na liczby zmiennoprzecinkowe.

W Pythonie numerycznym zwykle dostajemy od razu liczbę.

W SymPy najpierw powstaje obiekt matematyczny. Na tym obiekcie można robić uproszczenia, rozwinięcia, pochodne, całki i podstawienia.

In [ ]:
import math
# obliczenie numeryczne - surowy Python
display(math.sqrt(8))

# obliczenie symboliczne - SymPy
display(sym.sqrt(8))

Załóżmy, że masz funkcję $f(x,y) = x^3 \cdot (y - 1)$ i chcesz obliczyć jej wartość w punkcie $(2, 3)$.

Podstawiasz symbole liczb do zmiennych `x` oraz `y` i otrzymujesz symboliczny wynik (symbol liczby $16$)).

In [ ]:
x, y = sym.symbols('x y')
f = x**3 * (y - 1)

display( f.subs([(x, 2), (y, 3)]) ) # lista par (symbol, wartość)
display( f.subs( {x: 2, y: 3} )  )  # słownik symbol: wartość

`evalf()` służy do obliczenia przybliżenia numerycznego wyrażenia SymPy, czyli zamienia wynik dokładny na liczbę zmiennoprzecinkową.

Obliczmy wartość funkcji $f$ w punkcie $(\sqrt{2}, \frac{1}{3})$ - symbolicznie i numerycznie:

* symbolicznie

In [ ]:
display(f)
f1 = f.subs( {x: sym.sqrt(2), y: sym.Rational(1,3)} )
display(f1)

* symboliczno-numerycznie (symbole liczb zamieniane na liczby - wartości zmiennoprzecinkowe)

In [ ]:
f2 = f.evalf()
display(f2)

* numerycznie (przybliżenie zmiennoprzecinkowe)

In [ ]:
import math
f.evalf(subs={x: math.sqrt(2), y: 1/3})

`subs` a `evalf` - różnica jest taka:

- `subs(...)` - wykonuje podstawienie symboliczne
- `evalf()` - zamienia wynik na przybliżenie numeryczne

SymPy udostępnia też funkcję `N(expr)`, która jest skrótem do numerycznej ewaluacji.

In [ ]:
x = sym.Symbol('x')
expr = sym.sqrt(2) * sym.pi * x

display(expr)              # wynik symboliczny
display(expr.subs(x, 3))   # nadal dokładny, dla x = 3
display(sym.N(expr))       # symbole liczb na wartości liczbowe
display(expr.evalf(30, subs={x: 3}))  # wartość rzeczywista (30 cyfr) oraz podstawienie

## Symboliczne rozwiązywanie równań

W SymPy do symbolicznego rozwiązywania równań najczęściej używa się `solve()` i `solveset()`; pierwsza funkcja zwraca rozwiązania w formie wygodnej do dalszej pracy w Pythonie, a druga zwraca je jako obiekt zbioru matematycznego.

Jeśli chcesz rozwiązywać równania „po matematycznemu”, zwykle najlepszym punktem startu jest `solveset()`, zwłaszcza gdy ważna jest dziedzina albo nieskończenie wiele rozwiązań.

Równania podajemy w formie wyrażenia, np. `x**2 - y` albo jawnie jako obiekt `Eq(x**2, y)` (`Eq` - skrót od _equation_).

Przykład: rozwiążmy równianie $x^2 = y$ ze względu na $x$

In [ ]:
x, y = sym.symbols('x y')
rownanie = sym.Eq(x**2, y)
sym.solveset(rownanie, x, domain=sym.S.Reals)

`solve()` zwraca obiekt Pythona - listę albo słownik - które później można przetwarzać, np. wykonywać podstawienia.

In [ ]:
x, y = sym.symbols('x y')
wyniki = sym.solve(x**2 - y, x, dict=True)
print(wyniki)

**Przykład praktyczny**: dla $sin(x) = 0$ funkcja `solveset()` potrafi zapisać pełny zbiór rozwiązań okresowych, podczas gdy `solve()` zwraca tylko reprezentatywną skończoną listę.

In [ ]:
x = sym.symbols('x')

display( sym.solveset(sin(x), x) )
display( sym.solve(sin(x), x) )

Domyślnie `solveset()` rozwiązuje równania w dziedzinie liczb zespolonych, więc bez podania dziedziny możesz dostać także rozwiązania urojone.

Jeśli interesują Cię tylko liczby rzeczywiste, podaj `domain=sym.S.Reals`

In [ ]:
sym.solveset(x**2 + 1, x)

In [ ]:
sym.solveset(x**2 + 1, x, domain=sym.S.Reals)

Możemy wykorzystać SymPy do symbolicznego rozwiązywania równań. 

Przykład - równanie kwadratowe:

$$a x ^ 2 + b x + c = 0$$

In [ ]:
# deklaracja zmiennych symbolicznych dla potrzeb SymPy
a, b, c, x = sym.symbols('a b c x')

Metoda `solveset` pakietu `sympy`. Pierwszy argument jest wyrażeniem, dla którego wyznaczać będziemy pierwiastki (czyli porównywać do $0$). Drugi argument - zmienna według której wyznaczane będzie rozwiązanie.

In [ ]:
sym.solveset(a * x ** 2 + b * x + c, x)

:::{admonition} Ćwiczenie 2
Użyj SymPy do wyznaczenia rozwiązania równania stopnia trzeciego:

$$a x ^ 3 + b x ^ 2 + c  x + d = 0$$
:::

In [ ]:
# wpisz kod

Dziedziną dla `solveset()` może być dowolny zbiór.

Najczęściej używa się pełnych zbiorów liczbowych albo przedziałów, gdy chcesz zawęzić obszar szukania rozwiązań.

| Dziedzina      | Co oznacza                                                                                    | Przykład użycia                                       |
| -------------- | --------------------------------------------------------------------------------------------- | ----------------------------------------------------- |
| `S.Complexes`    | Wszystkie liczby zespolone; to jest domyślna dziedzina `solveset()`.                    | `solveset(x**2 + 1, x, domain=S.Complexes)`       |
| `S.Reals`        | Wszystkie liczby rzeczywiste.                                                           | `solveset(x**2 + 1, x, domain=S.Reals)`           |
| `S.Integers`     | Wszystkie liczby całkowite; można użyć, gdy interesują Cię tylko rozwiązania całkowite. | `solveset(x**2 - 4, x, domain=S.Integers)`        |
| `Interval(a, b)` | Przedział liczbowy, który ogranicza rozwiązania do wskazanego zakresu.                | `solveset(sin(x), x, domain=Interval(0, 2*pi))` |

In [ ]:
sym.solveset(x ** 2 + 1, x, domain=sym.S.Reals)

:::{admonition} Ćwiczenie 3
Wykorzystaj SymPy do rozwiązania poniższych równań:

- $x ^ 2 = 2$ w zbiorze $\mathbb{N}$
- $x ^ 3 + 2 x = 0$ w zbiorze $\mathbb{R}$
:::

In [ ]:
# wpisz kod

:::{admonition} Zadania

1. Rozwiąż równanie $x^4 - 5x^2 + 4 = 0$ w zbiorze `S.Reals`.

2. Rozwiąż równanie $\sqrt{x + 5} = x - 1$ w zbiorze `S.Reals`.

3. Rozwiąż równanie $\sin(2x) = \frac{\sqrt{3}}{2}$ w przedziale `Interval(0, 2\pi)`.

4. Rozwiąż równanie $x + \frac{6}{x} = 5$ w zbiorze `S.Integers`.

5. Rozwiąż równanie $e^x = 1$ w zbiorze `S.Complexes`.
:::

In [ ]:
# wpisz kod

## Układy równań

Dla układów liniowych SymPy udostępnia `linsolve()`, które zwraca rozwiązania jako zbiór uporządkowanych krotek i obsługuje także układy niedookreślone lub sprzeczne.

Dla układów nieliniowych służy `nonlinsolve()`, które może zwracać zarówno rozwiązania rzeczywiste, jak i zespolone.

In [ ]:
x, y = sym.symbols('x y')
sym.linsolve([x + y - 3, 2*x - y], [x, y])

In [ ]:
x, y = sym.symbols('x y')
sym. nonlinsolve([x*y - 1, x + y - 3], [x, y])

`linsolve()` i `nonlinsolve()` działają inaczej niż `solveset()` - są wywoływane z układem równań i listą niewiadomych, bez osobnego parametru `domain=...`, podczas gdy `solveset()` ma jawny parametr domain.

- `nonlinsolve()` może zwracać zarówno rozwiązania rzeczywiste, jak i zespolone.
- dla układów liniowych lepiej używać `linsolve()` niż `nonlinsolve()`

Jeśli dla `nonlinsolve()` chcesz tylko rozwiązania rzeczywiste, zalecane podejście to przefiltrowanie wyniku albo przecięcie go z odpowiednim zbiorem rozwiązań rzeczywistych, zamiast oczekiwać parametru `domain`.

> W prostych przypadkach założenia typu `symbols('a b', real=True)` mogą pomóc, ale nie należy ich traktować jako odpowiednika jawnej dziedziny z `solveset()`.

Przykład: rozwiązujesz układ równań:

$$
\begin{cases}
x^2 - 1 = 0 \\
y^2 + x = 0
\end{cases}
$$

w liczbach rzeczywistych.

Otrzymane pary $(x, y)$ przecinasz ze zbiorem $\mathbb{R^2}$

In [ ]:
from sympy import symbols, S, nonlinsolve

x, y = symbols('x y')

sol = nonlinsolve([x**2 - 1, y**2 + x], [x, y])
display("Wszystkie rozwiązania:", sol)

real_sol = sol & (S.Reals**2)
display("Tylko rzeczywiste:", real_sol)

:::{admonition} Ćwiczenie
Rozwiąż układ równań w liczbach rzeczywistych:

$$
\begin{cases}
x + y - 2 = 0 \\
x^2 + z - 5 = 0 \\
y - z = 0
\end{cases}
$$
:::

In [ ]:
# wpisz kod

## Analiza matematyczna

Obliczanie granicy funkcji, np.:

$$\lim_{x\to 0^+}\frac{1}{x}$$

In [ ]:
sym.limit(1/x, x, 0, dir="+")

:::{admonition} Ćwiczenie 4
Oblicz granice:

1. $\lim_{x\to 0^-}\frac{1}{x}$
2.  $\lim_{x\to 0}\frac{1}{x^2}$
:::

In [ ]:
# wpisz kod

Obliczanie pochodnych, np.:

$$\left( x ^ 2 - \cos(x) \right)'$$

In [ ]:
sym.diff(x ** 2 - sym.cos(x), x)

Obliczanie całek nieoznaczonych, np. $∫x^2-\cos x\;dx$:

In [ ]:
sym.integrate(x ** 2 - sym.cos(x), x)

Obliczanie całek oznaczonych, np. $$∫_{0}^{5}  x^2-\cos x\;dx$$

In [ ]:
wynik = sym.integrate(x ** 2 - sym.cos(x), (x, 0, 5))
wynik

In [ ]:
float(wynik)

:::{admonition} Ćwiczenie 5
Użyj SymPy do obliczenia:

1. $\frac{d\sin(x ^2)}{dx}$
2. $\frac{d(x ^2 + xy - \ln(y))}{dy}$
3. $\int e^x \cos(x)\;dx$
4. $\int_0^5 e^{2x}\;dx$
:::

In [ ]:
# wpisz kod

## Wykresy z SymPy

W SymPy wykresy robi się modułem `sympy.plotting`, który obsługuje wykresy 2D i 3D, a standardowo korzysta z backendu [matplotlib](http://matplotlib.org/).

Najważniejsze funkcje to `plot`, `plot_parametric`, `plot3d`, `plot3d_parametric_line`, `plot3d_parametric_surface` oraz `plot_implicit`.

Najprostszy wykres funkcji jednej zmiennej zrobisz przez `plot(expr, (x, a, b))`.

Jeśli nie podasz zakresu, SymPy użyje domyślnego przedziału dla zmiennej.

In [ ]:
from sympy import symbols, sin
from sympy.plotting import plot

x = symbols('x')
plot(sin(x), (x, -2, 2))

Możesz też rysować kilka funkcji naraz w jednym układzie współrzędnych.

In [ ]:
from sympy import symbols, sin, cos
from sympy.plotting import plot

x = symbols('x')
plot(sin(x), cos(x), (x, -2, 2))

:::{admonition} Ćwiczenie 6
Narysuj wykresy funkcji:

- $y=x + cos(x)$
- $y=x ^ 2 - e^x$ (być może przyda się `ylim` jako argument)

Eksperymentuj z wykresami.
:::

In [ ]:
# wprowadź kod

`plot_parametric()` służy do wykresów parametrycznych 2D, a `plot3d()` do powierzchni 3D.

Dla obiektów parametrycznych 3D są osobne funkcje do linii i powierzchni.

In [ ]:
from sympy import symbols, sin, cos
from sympy.plotting import plot_parametric, plot3d

t = symbols('t')
x, y = symbols('x y')

plot_parametric(cos(t), sin(t), (t, 0, 6))
plot3d(x**2 + y**2, (x, -2, 2), (y, -2, 2))

`plot_implicit()` - funkcja ta potrafi rysować także nierówności i warunki logiczne, na przykład regiony opisane przez `And(...)`.

In [ ]:
from sympy import symbols, Eq
from sympy.plotting import plot_implicit

x, y = symbols('x y')
plot_implicit(Eq(x**2 + y**2, 5))

`plot()` rysuje wykresy typu $y=f(x)$, a `plot_implicit()` rysuje równania i nierówności w układzie $(x,y)$, więc na przykład proste poziome i ukośne najwygodniej dodaje się jako kolejne funkcje, a zbiory typu półpłaszczyzna rysuje się przez nierówności.

Do bardziej „rysunkowych” dodatków SymPy pozwala też pobrać obiekt Matplotlib z backendu i wtedy korzystać z pełnych możliwości `ax.plot`, `ax.axvline` czy `ax.fill_between`.

`plot(expr, (x, a, b))` służy do rysowania funkcji jednej zmiennej jako krzywej, a w jednym wywołaniu możesz podać kilka wyrażeń na tym samym zakresie.

Jeśli chcesz później coś dołożyć, ustaw `show=False`, bo wtedy dostajesz obiekt `Plot`, który można dalej modyfikować albo łączyć z innymi seriami przez `append()` i `extend()`.

In [ ]:
from sympy import symbols, sin
from sympy.plotting import plot

x = symbols('x')
p = plot(sin(x), (x, -6, 6), show=False, legend=True)
p.show()

Asymptotę poziomą albo ukośną najwygodniej dodać jako kolejną funkcję w `plot()`, bo ten mechanizm obsługuje wiele wyrażeń naraz.

Styczną tworzysz dokładnie tak samo: liczysz punkt styczności i pochodną, budujesz równanie prostej stycznej, a potem rysujesz funkcję i tę prostą razem.

In [ ]:
from sympy import symbols, diff
from sympy.plotting import plot

x = symbols('x')
f = x**3 - 3*x + 1

x0 = 1
y0 = f.subs(x, x0)
m = diff(f, x).subs(x, x0)
tangent = y0 + m*(x - x0)

# przykładowa asymptota pozioma y = 2
asym_h = 2

p = plot(
    f,
    tangent,
    asym_h,
    (x, -3, 3),
    show=False,
    legend=True
)

p[0].label = "f(x)"
p[1].label = "styczna w x0=1"
p[2].label = "y=2"
p.show()

Dla asymptoty pionowej samo `plot()` nie jest najlepsze, bo ta funkcja jest przeznaczona do wykresów $y=f(x)$.

Pionową prostą wygodniej narysować przez `plot_implicit()` jako równanie typu `x - a`, ponieważ dokumentacja pokazuje, że przy wyrażeniu jednowymiarowym można jawnie wskazać `x_var` albo `y_var`.

In [ ]:
from sympy import symbols
from sympy.plotting import plot_implicit

x, y = symbols('x y')

plot_implicit(x - 1, x_var=(x, -4, 4), y_var=(y, -5, 5))

`plot_implicit()` służy nie tylko do równań, ale też do nierówności i regionów, więc półpłaszczyznę rysujesz bezpośrednio przez warunek typu $y > x + 1$ albo $x <= 2$.

Dokumentacja pokazuje też, że można łączyć warunki logiczne przez `And(...)`, co jest wygodne przy zaznaczaniu części wspólnej kilku zbiorów.

In [ ]:
from sympy import symbols, And
from sympy.plotting import plot_implicit

x, y = symbols('x y')

# półpłaszczyzna nad prostą y = x + 1
plot_implicit(y > x + 1, (x, -5, 5), (y, -5, 5))

# część wspólna dwóch półpłaszczyzn
plot_implicit(And(y > x, y > -x), (x, -5, 5), (y, -1, 5))

Aby jednocześnie narysować wykres funkcji $y=f(x)$ oraz asymptoty pionowe, najwygodnieszy sposób to narysować funkcję przez `plot(..., show=False)`, a pionowe asymptoty dodać już na osi Matplotlib przez ax.axvline(...), bo SymPy renderuje wykresy z backendem matplotlib, a zalecanym sposobem dodawania własnych elementów jest użycie `p._backend.ax`.

Prostszym rozwiązaniem jest pominąć dostęp do backendu i złożyć rysunek wyłącznie narzędziami SymPy: `plot()` dla funkcji, `plot_implicit()` dla pionowych prostych oraz `extend()` do połączenia wszystkiego na jednym wykresie.

Poniżej funkcja $f(x)= \frac{x+1}{x^2 - 1}$ ma asymptoty pionowe w $x=−1$ i $x=1$, więc wykres warto narysować w trzech osobnych przedziałach, aby nie łączyć gałęzi przez miejsca nieciągłości.

In [ ]:
from sympy import symbols
from sympy.plotting import plot, plot_implicit

x, y = symbols('x y')
f = 1 / (x**2 - 1)

# wykres funkcji w trzech kawałkach, żeby nie przechodzić przez asymptoty
p = plot(
    (f, (x, -4, -1.05)),
    (f, (x, -0.95, 0.95)),
    (f, (x, 1.05, 4)),
    show=False,
    ylim=(-10, 10),
    legend=True
)

p[0].label = "f(x)"
p[1].label = "f(x)"
p[2].label = "f(x)"

# pionowe asymptoty x = -1 i x = 1
a1 = plot_implicit(
    x + 1,
    x_var=(x, -4, 4),
    y_var=(y, -10, 10),
    line_color="red",
    show=False
)

a2 = plot_implicit(
    x - 1,
    x_var=(x, -4, 4),
    y_var=(y, -10, 10),
    line_color="red",
    show=False
)

# połączenie wszystkich serii na jednym rysunku
p.extend(a1)
p.extend(a2)

p.show()

Przy pionowych prostych nadal warto jawnie podać zakresy x_var i y_var, żeby wykres miał ten sam układ osi co funkcja.

Kolejny przykład. Weźmy funkcję $f(x)= \frac{x^2 - 1}{x^2 - x}$, która ma asymptotę pionową $x=0$, asymptotę poziomą $y=1$ oraz usuwalną nieciągłość w punkcie $(1,2)$.

In [ ]:
from sympy import symbols, Eq
from sympy.plotting import plot, plot_implicit

x, y = symbols('x y')
f = (x**2 - 1) / (x**2 - x)

p = plot(
    (f, (x, -4, -0.05)),
    (f, (x, 0.05, 0.95)),
    (f, (x, 1.05, 4)),
    show=False,
    ylim=(-4, 4),
    xlim=(-4, 4),
    legend=True,
    annotations=[
        {
            "text": "x = 0",
            "xy": (0, 3.2),
            "xytext": (0.3, 3.4)
        },
        {
            "text": "y = 1",
            "xy": (3.0, 1.0),
            "xytext": (2.2, 1.35)
        },
        {
            "text": "(1, 2) - dziura",
            "xy": (1.0, 2.0),
            "xytext": (1.35, 2.35)
        }
    ],
    markers=[
        # małe pełne punkty pomocnicze przy podpisach asymptot
        {
            "args": [[0], [3.2], "ro"],
            "markersize": 4
        },
        {
            "args": [[3.0], [1.0], "go"],
            "markersize": 4
        },

        # obwódka "dziury" w punkcie (1, 2)
        {
            "args": [[1.0], [2.0], "o"],
            "markerfacecolor": "white",
            "markeredgecolor": "blue",
            "markersize": 8
        }
    ]
)

p[0].label = "f(x)"
p[1].label = "f(x)"
p[2].label = "f(x)"

av = plot_implicit(
    Eq(x, 0),
    x_var=(x, -4, 4),
    y_var=(y, -4, 4),
    line_color="red",
    show=False
)

ah = plot_implicit(
    Eq(y, 1),
    x_var=(x, -4, 4),
    y_var=(y, -4, 4),
    line_color="green",
    show=False
)

p.extend(av)
p.extend(ah)

p.show()

## Wykres dla danych próbkowanych (Matplotlib)

Wykres dla danych próbkowanych (`numpy` - numeryczny Python, `linspace` - próbkowanie liniowe wartości z podanego zakresu).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(-40, 40, 1000) # w przedziale [-40, 40] wygeneruj 1000 punktów równo oddalonych od siebie
y = x ** 2 + 3 * x + 1         # dla listy punków `x` wyznacz listę wartości `y`
plt.figure()                   # przygotuj wykres
plt.plot(x, y);                # kreśl wykres

In [ ]:
# nakładanie wykresów
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(-10, 10, 1000)
f = np.cos(x)
g = np.sin(x)
plt.figure()
plt.plot(x, f)
plt.plot(x, g, color="red");

In [ ]:
# dodawanie tytułu
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(-10, 10, 1000)
f = np.cos(x)
plt.figure()
plt.plot(x, f)
plt.title("Wykres cos(x)");

In [ ]:
# uzywanie LaTeX w opisach
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(-10, 10, 1000)
f = np.sin(x)/x
plt.figure()
plt.plot(x, f)
plt.title("Wykres: $\\frac{sin(x)}{x}$");

In [ ]:
# nazwy osi
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(-10, 10, 1000)
f = np.cos(x)
plt.figure()
plt.plot(x, f)
plt.xlabel("$x$")
plt.ylabel("$y$");

In [ ]:
# dodawanie legendy
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(-10, 10, 1000)
f = np.cos(x)
g = np.sin(x)
plt.figure()
plt.plot(x, f, label="$cos(x)$")
plt.plot(x, g, label="$sin(x)$", color="red")
plt.legend();

In [ ]:
# dane empiryczne
import matplotlib.pyplot as plt
import numpy as np

x = np.array((1, 2, 3, 4, 5))
y = np.array((-1, -2, 4, -1, 7))
plt.figure()
plt.scatter(x, y)
plt.show()

Zbiór wielu przykładów konstruowania wykresów w `matplotlib`: <https://matplotlib.org/stable/gallery/>